In [5]:
# =========================
# Build DAILY liquidity + size factors
# company first, then date ascending
# =========================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# =====================================================
# 1. File paths
# =====================================================
INPUT_PATH = r"D:\Cornell University\Term 3\SYSEN 5270\project\stock_data (1).csv"
OUTPUT_PATH = r"D:\Cornell University\Term 3\SYSEN 5270\project\daily_liquidity_size_factor_panel_company_first.xlsx"

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"Cannot find input file:\n{INPUT_PATH}")

print("INPUT_PATH =", INPUT_PATH)
print("OUTPUT_PATH =", OUTPUT_PATH)

# =====================================================
# 2. Read raw file
# =====================================================
raw = pd.read_csv(INPUT_PATH)

print("\nRaw shape:", raw.shape)
print("\nFirst 20 columns:")
print(raw.columns[:20].tolist())
print("\nHead:")
print(raw.head())

# =====================================================
# 3. Find date column
# =====================================================
date_col = None
for c in raw.columns:
    if "date" in c.lower():
        date_col = c
        break

if date_col is None:
    date_col = raw.columns[0]

raw = raw.rename(columns={date_col: "date"})
raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"]).copy()

print("\nDate column used:", "date")
print("Date range:", raw["date"].min(), "to", raw["date"].max())

# =====================================================
# 4. Detect columns for each field
#    Expected patterns:
#    open_AAPL, high_AAPL, low_AAPL, adjclose_AAPL,
#    volume_AAPL, turnover_AAPL, market_cap_AAPL
# =====================================================
def collect_field_map(columns, prefix):
    out = {}
    prefix_lower = prefix.lower()
    for c in columns:
        cl = c.lower()
        if cl.startswith(prefix_lower + "_"):
            ticker = c[len(prefix) + 1:]
            out[ticker] = c
    return out

open_map = collect_field_map(raw.columns, "open")
high_map = collect_field_map(raw.columns, "high")
low_map = collect_field_map(raw.columns, "low")
adjclose_map = collect_field_map(raw.columns, "adjclose")
volume_map = collect_field_map(raw.columns, "volume")
turnover_map = collect_field_map(raw.columns, "turnover")
marketcap_map = collect_field_map(raw.columns, "market_cap")

tickers = sorted(
    list(
        set(open_map.keys())
        & set(high_map.keys())
        & set(low_map.keys())
        & set(adjclose_map.keys())
        & set(volume_map.keys())
        & set(turnover_map.keys())
        & set(marketcap_map.keys())
    )
)

if len(tickers) == 0:
    raise ValueError(
        "Could not detect matching ticker columns for open/high/low/adjclose/volume/turnover/market_cap.\n"
        "Please print raw.columns.tolist() and check naming patterns."
    )

print("\nNumber of usable tickers:", len(tickers))
print("Tickers:")
print(tickers)

# =====================================================
# 5. Convert wide daily data to long daily panel
# =====================================================
long_frames = []

for ticker in tickers:
    tmp = raw[
        [
            "date",
            open_map[ticker],
            high_map[ticker],
            low_map[ticker],
            adjclose_map[ticker],
            volume_map[ticker],
            turnover_map[ticker],
            marketcap_map[ticker],
        ]
    ].copy()

    tmp.columns = [
        "date",
        "open",
        "high",
        "low",
        "adjclose",
        "volume",
        "turnover",
        "market_cap",
    ]
    tmp["company"] = ticker

    numeric_cols = ["open", "high", "low", "adjclose", "volume", "turnover", "market_cap"]
    for col in numeric_cols:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    long_frames.append(tmp)

daily_df = pd.concat(long_frames, ignore_index=True)

# Remove missing / invalid rows
required_cols = ["date", "company", "open", "high", "low", "adjclose", "volume", "turnover", "market_cap"]
daily_df = daily_df.dropna(subset=required_cols).copy()

for col in ["open", "high", "low", "adjclose", "volume", "turnover", "market_cap"]:
    daily_df.loc[~np.isfinite(daily_df[col]), col] = np.nan

daily_df = daily_df.dropna(subset=["open", "high", "low", "adjclose", "volume", "turnover", "market_cap"]).copy()

print("\nDaily long data shape:", daily_df.shape)
print("\nDaily long sample:")
print(daily_df.head())

# =====================================================
# 6. Build daily variables
# =====================================================
daily_df["Dollar Volume"] = daily_df["adjclose"] * daily_df["volume"]
daily_df["Turnover"] = daily_df["turnover"]
daily_df["Market Cap"] = daily_df["market_cap"]

for col in ["Dollar Volume", "Turnover", "Market Cap"]:
    daily_df.loc[~np.isfinite(daily_df[col]), col] = np.nan

daily_df = daily_df.dropna(subset=["Dollar Volume", "Turnover", "Market Cap"]).copy()

daily_df["alpha5"] = np.log1p(daily_df["Dollar Volume"])
daily_df["alpha6_base"] = np.log1p(daily_df["Market Cap"])

# =====================================================
# 7. Winsorize + cross-sectional z-score by DATE
# =====================================================
def winsorize_series(s, lower=0.01, upper=0.99):
    s = s.copy()
    if s.notna().sum() == 0:
        return s
    q_low = s.quantile(lower)
    q_high = s.quantile(upper)
    return s.clip(lower=q_low, upper=q_high)

def zscore_series(s):
    s = s.copy()
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

daily_df["alpha5"] = daily_df.groupby("date")["alpha5"].transform(winsorize_series)
daily_df["Turnover"] = daily_df.groupby("date")["Turnover"].transform(winsorize_series)
daily_df["alpha6_base"] = daily_df.groupby("date")["alpha6_base"].transform(winsorize_series)

daily_df["alpha5_z"] = daily_df.groupby("date")["alpha5"].transform(zscore_series)
daily_df["alpha_turnover_z"] = daily_df.groupby("date")["Turnover"].transform(zscore_series)
daily_df["alpha6"] = daily_df.groupby("date")["alpha6_base"].transform(zscore_series)

# Final liquidity factor
daily_df["alpha5"] = (daily_df["alpha5_z"] + daily_df["alpha_turnover_z"]) / 2

# =====================================================
# 8. Format date and final output
# =====================================================
daily_df["date"] = pd.to_datetime(daily_df["date"], errors="coerce")

final_df = daily_df[
    [
        "company",
        "date",
        "open",
        "high",
        "low",
        "adjclose",
        "volume",
        "Dollar Volume",
        "Turnover",
        "Market Cap",
        "alpha5",
        "alpha6",
    ]
].copy()

# Sort exactly as requested:
# first company, then date ascending within each company
final_df = final_df.sort_values(["company", "date"]).reset_index(drop=True)

# Date display format: m/d/yyyy
final_df["date"] = final_df["date"].dt.strftime("%#m/%#d/%Y")

# =====================================================
# 9. Save output
# =====================================================
final_df.to_excel(OUTPUT_PATH, index=False)

print("\nDone.")
print("Output saved to:", OUTPUT_PATH)
print("\nFinal shape:", final_df.shape)
print("\nColumns:")
print(final_df.columns.tolist())

print("\nSample:")
print(final_df.head(30))

print("\nMissing ratios:")
for c in final_df.columns:
    if c not in ["company", "date"]:
        print(f"{c}: {final_df[c].isna().mean():.2%}")

print("\nNumber of companies:", final_df["company"].nunique())
print("Number of dates:", final_df["date"].nunique())

INPUT_PATH = D:\Cornell University\Term 3\SYSEN 5270\project\stock_data (1).csv
OUTPUT_PATH = D:\Cornell University\Term 3\SYSEN 5270\project\daily_liquidity_size_factor_panel_company_first.xlsx

Raw shape: (2765, 169)

First 20 columns:
['date', 'open_AAPL', 'open_ABBV', 'open_AMZN', 'open_AVGO', 'open_BAC', 'open_BRK-B', 'open_COST', 'open_CVX', 'open_GOOG', 'open_GOOGL', 'open_JNJ', 'open_JPM', 'open_LLY', 'open_MA', 'open_META', 'open_MSFT', 'open_MU', 'open_NFLX', 'open_NVDA']

Head:
         date  open_AAPL  open_ABBV  open_AMZN  open_AVGO   open_BAC  \
0  2015-01-02  27.847500  65.440002    15.6290     10.093  17.990000   
1  2015-01-05  27.072500  65.500000    15.3505     10.007  17.790001   
2  2015-01-06  26.635000  65.620003    15.1120      9.895  17.420000   
3  2015-01-07  26.799999  64.570000    14.8750      9.705  17.139999   
4  2015-01-08  27.307501  68.160004    15.0160     10.053  17.160000   

   open_BRK-B   open_COST    open_CVX  open_GOOG  ...  market_cap_META  \